In [ ]:
import glob
import os
from pathlib import Path

import pandas as pd
import numpy as np
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer


1.テキストデータの読み込み

In [ ]:
dir_path = Path("../data")

list_file_paths = dir_path.glob("*.md")

In [ ]:
documents = []

for file_path in list_file_paths:

    print(file_path)
    content = file_path.read_text(encoding="utf-8")
    source = file_path.name
    project = source.split(".")[0].split("_")[1]

    documents.append(
        {
            "content": content,
            "source": source,
            "project": project
        }
    )

In [ ]:
# documentsの中身確認
documents

2.chromadbに格納

2.1chunk準備

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=100
)

In [ ]:
chunked_documents = []

# chunk_id = 0

for document in documents:

    chunks = splitter.split_text(
        document["content"]
    )

    for chunk in chunks:

        # プロジェクト名を擬似的にヘッダーにする
        chunk_with_header = f"""プロジェクト名: {document["project"]}
                            {chunk}"""

        chunked_documents.append(
            {
                # "content": chunk,
                "content": chunk_with_header,
                "source": document["source"],
                "project": document["project"]
                # "chunk_id":chunk_id
            }
        )
        # chunk_id += 1

In [ ]:
# 中身確認
chunked_documents[:3]

2.2chromadbのcollectionにadd

In [ ]:
# クライアントの初期化。ローカルにデータを保存
client = chromadb.PersistentClient(path="./chroma_db")

In [ ]:
# コレクション（テーブルのようなもの）作成

# 既存コレクションの削除
try:
    client.delete_collection(name="portfolio_rag")
except ValueError:
    pass # まだ存在しない場合は無視

collection = client.get_or_create_collection(
    name="portfolio_rag",
    metadata={
        "hnsw:space": "cosine"
    }
    )

In [ ]:
# embedding用のモデルを設定
model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

In [ ]:
for id, doc in enumerate(chunked_documents):
    
    # chunkした文章をembedding化
    embedding = model.encode(doc["content"])
    
    collection.add(
        documents=[doc["content"]],
        embeddings=[embedding.tolist()],
        metadatas=[
            {
                "source": doc["source"],
                "chunk_id": id,
                "project": doc["project"]
            }
        ],
        ids=[f"chunk_{id}"]
    )

3.chromadbで検索

In [ ]:
query = "太陽光発電量予測で使ったモデルを教えて"
query_embedding = model.encode(query)

result = collection.query(
    query_embeddings=[query_embedding.tolist()],
    n_results=10
)

In [ ]:
for rank, (document, metadata, distance) in enumerate(zip(result["documents"][0],result["metadatas"][0],result["distances"][0]
    ),start=1):

    print("=" * 10)
    print(f"検索順位:{rank}位")
    print(f"source : {metadata['source']}")
    # print(f"distance : {distance:.3f}") # chromadbのデフォルトがdisatance
    print(f"similarity: {1-distance:.3f}") #コレクション作成時に、コサイン類似度に変更した場合はこっち
    print(document[:100])